In [2]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.table import Table
from astropy.cosmology import Planck18 as cosmo
import astropy.units as u

from tools.functions import (completeness_function,
                             radec_z_to_cartesian,
                             knn_density
                            )

In [ ]:
catalog_path = "/data/hetdex/u/bgrashey/data_/full_catalog.fits"
cat = Table.read(catalog_path)

# Add the completeness to each catalog source

In [ ]:
Flux = np.asanyarray(cat["FLUX"]) / 1e-17

popt = np.array([0.43107471, 0.53873367, 2.32606927, 0.50485321])

cat["COMPLETENESS"] = completeness_function(Flux, *popt)

# Add the Luminosity to eah catalog source

In [ ]:
Flux = np.asanyarray(cat["FLUX"])  # 10 für W/m^2
z = np.asanyarray(cat["REDSHIFT"])

def luminosity(F, z):
    D_L = cosmo.luminosity_distance(z).to(u.cm)
    L = 4 * np.pi * D_L**2 * F
    return L.value

cat["LUMINOSITY"] = luminosity(Flux, z)

# Add the local density to each source

In [ ]:
pos = radec_z_to_cartesian(cat["ra_vdfi"], cat["dec_vdfi"], cat["z_vdfi"])
rho = knn_density(pos)

cat["DENSITY"] = rho

# Inspect the catalog

In [ ]:
print(cat.columns)

In [ ]:
LAE_mask = cat["PROB"] > 0.5
cat = cat[LAE_mask]
cat.sort("FLUX")

In [ ]:
redshift_bins = np.linspace(2.7, 3.4, 15)

plt.hist(cat["REDSHIFT"], bins=redshift_bins, edgecolor="black")
plt.xlabel("z")
plt.ylabel("N")
plt.show()

In [ ]:
prob_bins = np.linspace(0,1,11)
plt.hist(cat["PROB"], bins=prob_bins, edgecolor="black")
plt.xlabel("P(LAE)")
plt.ylabel("N")
plt.show()

# Inspect the LAE catalog with narrow band images

In [ ]:
from tools.functions import plot_cutout_grid

plot_cutout_grid(
    catalog=cat,
    cube_file="/data/hetdex/u/bgrashey/cubes/ssa22_fullfp_stack.fits",
    output_file="./nb_images.pdf",
    num_cutouts=len(cat),
    num_wave_slices=10,
)